why does Pydantic exist?
-

Python doesn’t check data types. Look at this:

In [ ]:
age = "twenty"
age = 20
age = None

Python accepts all of these. That’s fine for small scripts, but real programs receive data from outside: web forms, APIs, JSON files, databases. That data is often messy:

A user types "abc" where you expected a number

An API forgets to send a field

A date arrives as text

If bad data enters your program, it crashes later in a confusing place, or worse, silently produces wrong results.

Pydantic is a library that checks your data at the door. You describe what your data should look like, and Pydantic validates it, converts it where sensible, and raises a clear error if it’s wrong.

Install and import:

In [ ]:
!pip install -q pydantic

In [ ]:
import pydantic
print(pydantic.__version__)

The problem without Pydantic:

In [ ]:
def create_user(name, age):
    return {"name": name, "age": age}

user = create_user("Ali", "twenty")
print(user)

In [ ]:
print(user["age"] + 1)

First Pydantic model:

In [ ]:
from pydantic import BaseModel

class User(BaseModel):
    name: str
    age: int

BaseModel is Pydantic’s base class. Inheriting from it gives our class superpowers.

name: str is a type hint: “name must be a string.”

age: int means “age must be an integer.”

We call this a model (a blueprint for data).

Create valid data:

In [ ]:
user = User(name="Ali", age=20)

print(user)
print(user.name)
print(user.age)
print(type(user))

Create invalid data:

In [ ]:
user = User(name="Ali", age="twenty")

The error happens at the entrance, not 50 lines later.

In [ ]:
user = User(name="Ali")

By default, every field is required.

Exercise:
-

Create a Book model with title (str), pages (int), and price (float). Then:

Create one valid book.

Try pages="many" and read the error.

Try leaving out price.

In [ ]:
class Book(BaseModel):
    title: str
    pages: int
    price: float

book = Book(title="Python 101", pages=300, price=19.99)
print(book)

Book(title="Python 101", pages="many", price=19.99)  # error
Book(title="Python 101", pages=300)                  # error: price required

Types, coercion and validation errors:
-

Pydantic doesn’t just check, it converts

age="twenty" failed. But what about age="20"? That’s a string, not an int, but it looks like a number. Let’s see what Pydantic does.

This behavior is called coercion: Pydantic tries to convert the input to the declared type when the conversion is safe and obvious.

Setup:

In [ ]:
from pydantic import BaseModel, ValidationError

class User(BaseModel):
    name: str
    age: int

We also imported ValidationError

Coercion in action:

In [ ]:
user = User(name="Ali", age="20")

print(user)
print(user.age)
print(type(user.age))

In [ ]:
print(User(name="Ali", age=20.0))
print(User(name="Ali", age=" 25 "))
print(User(name="Ali", age=True))

Python treats True as 1, so Pydantic accepts it.

What is NOT coerced:

In [ ]:
try:
    User(name="Ali", age=20.5)
except ValidationError as e:
    print(e)

20.0 converts safely to 20, but 20.5 would lose information, so Pydantic refuses.

Catching errors with try/except:

In real programs, you don’t want a crash. You want to catch the error and handle it. That’s what ValidationError is for:

In [ ]:
try:
    user = User(name="Ali", age="twenty")
except ValidationError as e:
    print("Oops, bad data!")
    print(e)

The program keeps running. This is how you’d show a friendly message to a user instead of crashing.

Reading errors like a programmer:

The printed error is for humans. If you need to work with errors in code, use .errors():

In [ ]:
try:
    User(name=123, age="twenty")
except ValidationError as e:
    for error in e.errors():
        print("Field:  ", error["loc"])
        print("Problem:", error["msg"])
        print("Type:   ", error["type"])
        print("---")

Pydantic reports ALL errors at once, not just the first one. Great for forms: you can show the user every mistake in one go.

Each error is a dictionary with loc (where), msg (what), and type (error code).

Models aren’t limited to str and int:

In [ ]:
from datetime import date

class Student(BaseModel):
    name: str
    gpa: float
    is_active: bool
    birthday: date

s = Student(
    name="Sara",
    gpa="3.8",             # str -> float
    is_active="yes",       # str -> bool
    birthday="2004-05-17"  # str -> date
)

print(s)
print(s.birthday.year)

"yes" became True, and the date string became a real date object, so s.birthday.year works. Pydantic accepts these as true: True, 1, "true", "yes", "on", "1", "t", "y", and these as false: False, 0, "false", "no", "off", "0", "f", "n".

Exercise:
-

Create a Product model with name (str), price (float), in_stock (bool), and added_on (date). Then:

Create one using only strings for the values: "9.99", "true", "2025-01-15".

Print type(product.price) and product.added_on.year.

Try price="free" inside a try/except and print each error’s loc and msg.

In [ ]:
from datetime import date

class Product(BaseModel):
    name: str
    price: float
    in_stock: bool
    added_on: date

p = Product(name="Pen", price="9.99", in_stock="true", added_on="2025-01-15")
print(type(p.price))
print(p.added_on.year)

try:
    Product(name="Pen", price="free", in_stock="true", added_on="2025-01-15")
except ValidationError as e:
    for error in e.errors():
        print(error["loc"], error["msg"])

Optional fields, defaults, and Field() constraints:
-

real data is rarely complete or unlimited

So far, every field was required, and any int was accepted. Real life is different:

Some fields are optional (a user might not give a phone number).

Some fields have a sensible default (a new account starts as “active”).

Some values must stay in a range (age can’t be -5 or 500).

Default values:

Give a field a default with =

In [ ]:
from pydantic import BaseModel, ValidationError

class User(BaseModel):
    name: str
    age: int
    is_active: bool = True
    role: str = "student"

user = User(name="Ali", age=20)
print(user)

We never passed is_active or role, so Pydantic filled them in. Any field with a default becomes not required. You can still override it:

In [ ]:
print(User(name="Sara", age=22, role="teacher"))

Optional fields:

Sometimes “no value” is a valid answer. In Python, “no value” is None. We write that with | None:

In [ ]:
class User(BaseModel):
    name: str
    age: int
    phone: str | None = None

print(User(name="Ali", age=20))
print(User(name="Ali", age=20, phone="0300-1234567"))

an older Python (below 3.10), write Optional[str] instead, with from typing import Optional. Same meaning.

Constraints with Field():

Field() lets us add rules to a field:

In [ ]:
from pydantic import Field
from datetime import datetime

In [ ]:
class User(BaseModel):
    name: str = Field(min_length=2, max_length=50)

print(User(name="Ali"))

In [ ]:
try:
    User(name="A")
except ValidationError as e:
    for error in e.errors():
        print(error["loc"], "->", error["msg"])

Descriptions and required fields with Field:

In [ ]:
class User(BaseModel):
    name: str = Field(description="Full name of the user")
    age: int = Field(ge=0, le=120, description="Age in years")

print(User.model_json_schema())

The description doesn’t change validation. It shows up in generated documentation

When the default must be computed fresh each time (like the current time), use default_factory with Field:

In [ ]:
from pydantic import Field
from datetime import datetime

class Post(BaseModel):
    title: str
    created_at: datetime = Field(default_factory=datetime.now)

p1 = Post(title="Hello")
print(p1)

Exercise:
-

Create a Movie model with:

title: string, 1 to 100 characters

year: integer, between 1888 and 2100

rating: float between 0 and 10

director: optional string (default None)

Nested models and lists:
-

Real data is rarely flat. A user has an address. An order has a list of items. JSON from APIs looks like this:

In [ ]:
{
  "name": "Ali",
  "address": {"city": "Lahore", "country": "Pakistan"}
}

Here address is not a string or number, it’s a smaller piece of data with its own fields. In Pydantic, you handle this by putting one model inside another.

In [ ]:
from pydantic import BaseModel, Field, ValidationError

class Address(BaseModel):
    city: str
    country: str

class User(BaseModel):
    name: str
    address: Address     # a model used as a type!

The type of address is Address, our own model. Order matters: define Address before User, because User refers to it.

Create it from a dictionary:

In [ ]:
user = User(
    name="Ali",
    address={"city": "Lahore", "country": "Pakistan"}
)

print(user)
print(user.address)
print(user.address.city)
print(type(user.address))

Lists of simple type:

Before lists of models, lists of plain values:

In [ ]:
class Student(BaseModel):
    name: str
    scores: list[int]

s = Student(name="Ali", scores=[90, "85", 70.0])
print(s.scores)

list[int] means “a list where every item is an int”. Each item is validated and coerced separately: "85" became 85, 70.0 became 70.

Lists of models:

In [ ]:
class Item(BaseModel):
    name: str
    price: float = Field(gt=0)
    quantity: int = Field(ge=1, default=1)

class Order(BaseModel):
    order_id: int
    items: list[Item]

order = Order(
    order_id=1001,
    items=[
        {"name": "Pen", "price": 1.5, "quantity": 10},
        {"name": "Notebook", "price": "4.25"},
        {"name": "Bag", "price": 20, "quantity": 2},
    ]
)

print(order.items[1])
print(order.items[0].name)